# Ejecución en Google Colab

Este notebook ejecuta el proyecto [mlops-diabetes](https://github.com/tulipon/mlops-diabetes) de principio a fin en Colab:

1. Clona el repositorio e instala el entorno con `uv`.
2. Ejecuta los tests y el pipeline (entrenamiento, registro en MLflow y quality gate).
3. Arranca la API y la publica mediante el proxy de Colab.
4. Opcionalmente arranca la interfaz de MLflow.
5. Opcionalmente guarda el estado en Google Drive.

> **Usa el flujo nativo, no Docker.** Colab no ofrece un daemon de Docker fiable. El pipeline es pequeño y se ejecuta en CPU; no necesitas GPU.

Ejecuta las celdas en orden. La primera vez tarda unos minutos porque `uv` descarga Python 3.13 y las dependencias.

## 1. Clonar el repositorio

La celda es idempotente: si `/content/mlops-diabetes` ya existe, no vuelve a clonar.

In [ ]:
%%bash
REPO_DIR=/content/mlops-diabetes

if [ -d "$REPO_DIR/.git" ]; then
  echo "El repositorio ya existe en $REPO_DIR"
else
  git clone https://github.com/tulipon/mlops-diabetes.git "$REPO_DIR"
fi

In [ ]:
%cd /content/mlops-diabetes

## 2. Instalar el entorno

`make setup` crea `.venv` con Python 3.13 e instala las dependencias de runtime y desarrollo. Si Colab no trae esa versión, `uv` descarga un intérprete propio.

In [ ]:
!pip install -q uv
!make setup

## 3. Tests y pipeline

`make test` ejecuta la suite (33 tests). `make pipeline` descarga el dataset, lo valida y limpia, entrena los candidatos, registra los runs en MLflow y promociona el champion solo si cumple:

- RMSE de test < 54.0
- R² de test > 0.40

El resumen queda en `reports/pipeline_report.json` y el modelo se carga con el alias `champion`.

In [ ]:
!make test && make pipeline

## 4. Arrancar la API

`make api` usa `--reload` y deja la celda bloqueada, así que aquí lanzamos Uvicorn en segundo plano. La API carga el champion desde el MLflow local (`mlflow.db` + `mlruns/`); no hace falta arrancar el servidor de MLflow.

In [ ]:
%%bash
cd /content/mlops-diabetes

# Detén solo la instancia que arrancó este notebook.
if [ -f /tmp/api.pid ] && kill -0 "$(cat /tmp/api.pid)" 2>/dev/null; then
  kill "$(cat /tmp/api.pid)"
  sleep 1
fi

nohup .venv/bin/python -m uvicorn api.main:app \
  --host 0.0.0.0 --port 8000 > /tmp/api.log 2>&1 &
echo $! > /tmp/api.pid

for _ in $(seq 1 30); do
  if curl -sf http://127.0.0.1:8000/health > /dev/null 2>&1; then
    break
  fi
  sleep 1
done

if curl -sf http://127.0.0.1:8000/health; then
  echo
  echo "API lista (PID $(cat /tmp/api.pid))"
else
  echo "La API no respondió. Últimas líneas de /tmp/api.log:"
  tail -n 30 /tmp/api.log
  exit 1
fi

### Abrir la documentación

`serve_kernel_port_as_iframe` incrusta el puerto del kernel en la salida del notebook. Ejecuta la celda para ver Swagger UI. Si prefieres una pestaña completa, `serve_kernel_port_as_window(8000, path="/docs")` sigue disponible, aunque Colab lo marca como obsoleto.

In [ ]:
from google.colab.output import serve_kernel_port_as_iframe

serve_kernel_port_as_iframe(8000, path="/docs", height=600)

### Probar `/predict`

Petición de ejemplo con una fila del dataset. La respuesta de referencia es `210.74098559942396`.

In [ ]:
import json
import urllib.request

payload = {
    "age": 0.038076,
    "sex": 0.050680,
    "bmi": 0.061696,
    "bp": 0.021872,
    "s1": -0.044223,
    "s2": -0.034821,
    "s3": -0.043401,
    "s4": -0.002592,
    "s5": 0.019907,
    "s6": -0.017646,
}

request = urllib.request.Request(
    "http://127.0.0.1:8000/predict",
    data=json.dumps(payload).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST",
)
with urllib.request.urlopen(request) as response:
    print(json.dumps(json.load(response), indent=2))

## 5. Interfaz de MLflow (opcional)

Solo hace falta para explorar experimentos, runs y artefactos. La API no la necesita. La celda de arranque desactiva `X-Frame-Options` para poder incrustar la UI.

In [ ]:
%%bash
cd /content/mlops-diabetes

# Detén solo la instancia que arrancó este notebook.
if [ -f /tmp/mlflow.pid ] && kill -0 "$(cat /tmp/mlflow.pid)" 2>/dev/null; then
  kill "$(cat /tmp/mlflow.pid)"
  sleep 1
fi

nohup .venv/bin/python -m mlflow server \
  --backend-store-uri sqlite:///mlflow.db \
  --default-artifact-root ./mlruns \
  --host 0.0.0.0 --port 5000 --workers 1 \
  --allowed-hosts '*' --x-frame-options NONE > /tmp/mlflow.log 2>&1 &
echo $! > /tmp/mlflow.pid

for _ in $(seq 1 30); do
  if curl -sf http://127.0.0.1:5000/health > /dev/null 2>&1; then
    break
  fi
  sleep 1
done

if curl -sf http://127.0.0.1:5000/health > /dev/null 2>&1; then
  echo "MLflow listo (PID $(cat /tmp/mlflow.pid))"
else
  echo "MLflow no respondió. Últimas líneas de /tmp/mlflow.log:"
  tail -n 30 /tmp/mlflow.log
  exit 1
fi

### Abrir la interfaz

Ejecuta la celda para incrustar MLflow en el notebook.

In [ ]:
from google.colab.output import serve_kernel_port_as_iframe

serve_kernel_port_as_iframe(5000, height=600)

## 6. Guardar el estado en Google Drive (opcional)

El almacenamiento de Colab es efímero: al desconectar el runtime se pierden `.venv`, `mlflow.db`, `mlruns/` y `reports/`.

Para montar Drive, ejecuta esta celda y autoriza el acceso:

```python
from google.colab import drive
drive.mount("/content/drive")
```

Después, con la API detenida, ejecuta la celda siguiente. El estado se copia a `MyDrive/mlops-diabetes`.

In [ ]:
%%bash
BACKUP_DIR=/content/drive/MyDrive/mlops-diabetes
PROJECT_DIR=/content/mlops-diabetes

if [ ! -d /content/drive/MyDrive ]; then
  echo "Google Drive no está montado."
  echo "Ejecuta primero:"
  echo "  from google.colab import drive"
  echo "  drive.mount('/content/drive')"
  echo "y vuelve a lanzar esta celda."
  exit 0
fi

if [ -f /tmp/api.pid ] && kill -0 "$(cat /tmp/api.pid)" 2>/dev/null; then
  echo "Aviso: la API sigue en ejecución."
  echo "Deténla antes de copiar mlflow.db y vuelve a lanzar esta celda:"
  echo '  kill "$(cat /tmp/api.pid)"'
  exit 1
fi

mkdir -p "$BACKUP_DIR"
cp "$PROJECT_DIR/mlflow.db" "$BACKUP_DIR/"
rm -rf "$BACKUP_DIR/mlruns" "$BACKUP_DIR/reports"
cp -r "$PROJECT_DIR/mlruns" "$PROJECT_DIR/reports" "$BACKUP_DIR/"
du -sh "$BACKUP_DIR"

### Restaurar tras reiniciar el runtime

Vuelve a clonar el repositorio en la misma ruta, ejecuta `make setup` y copia el estado **antes** de lanzar `make pipeline` o la API:

```bash
cp /content/drive/MyDrive/mlops-diabetes/mlflow.db /content/mlops-diabetes/
rm -rf /content/mlops-diabetes/mlruns /content/mlops-diabetes/reports
cp -r /content/drive/MyDrive/mlops-diabetes/mlruns /content/mlops-diabetes/
cp -r /content/drive/MyDrive/mlops-diabetes/reports /content/mlops-diabetes/
```

El registry guarda rutas absolutas a los artefactos. Si restauras en otra ruta, vuelve a ejecutar `make pipeline` para regenerar el champion.

## 7. Solución de problemas

- **`make: command not found`**: instala `make` con `!apt-get -qq update && apt-get -qq install -y make`, o ejecuta los comandos equivalentes de la sección 3 del README.
- **`/health` devuelve `model_loaded: false` o `/model-info` responde 503**: el pipeline no promocionó ningún modelo. Revisa `reports/pipeline_report.json` y vuelve a ejecutar `make pipeline`.
- **Puerto 8000 ocupado**: cambia el puerto en la celda de Uvicorn y en `serve_kernel_port_as_iframe`.
- **La API o MLflow dejan de responder**: el runtime se ha reiniciado o ha expirado. Vuelve a ejecutar las celdas de arranque; los ficheros también se habrán perdido si no estaban en Drive.
- **Logs**: `/tmp/api.log` y `/tmp/mlflow.log`.

Para detener los servicios:

```bash
pkill -f "uvicorn api.main:app"
pkill -f "mlflow server"
```

> `google.colab.output` solo existe dentro de Colab. Si ejecutas el notebook en Jupyter local, abre directamente <http://localhost:8000/docs>.